# POSTER Grad-CAM Visualization

Grad-CAM trên IR-50 backbone của POSTER — xem model nhìn vào vùng nào trên khuôn mặt (mắt, miệng đỏ lên).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

print('PyTorch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ─── Import model components ───
from models.ir50 import Backbone
from models.mobilefacenet import MobileFaceNet
from models.hyp_crossvit import HyVisionTransformer

In [ ]:
# ─── POSTER model ───
class SE_block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x)


class POSTER(nn.Module):
    def __init__(self, num_classes=7, depth=8):
        super().__init__()
        self.face_landback = MobileFaceNet([112, 112], 136)
        self.ir_back = Backbone(50, 0.0, 'ir')
        self.ir_layer = nn.Linear(1024, 512)
        self.pyramid_fuse = HyVisionTransformer(
            in_chans=49, q_chanel=49, embed_dim=512,
            depth=depth, num_heads=8, mlp_ratio=2.0,
            drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
        )
        self.se_block = SE_block(512)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x_face = F.interpolate(x, size=112)
        _, x_face = self.face_landback(x_face)
        x_face = x_face.view(B, -1, 49).transpose(1, 2)
        x_ir = self.ir_layer(self.ir_back(x))
        y = self.se_block(self.pyramid_fuse(x_ir, x_face))
        y = self.dropout(y)
        return self.head(y), y

In [ ]:
# ─── Load checkpoint với weight mapping ───
import collections

def load_poster_weights(model, checkpoint_path, device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    sd = ckpt.get('state_dict', ckpt)

    # Direct load thử trước
    try:
        model.load_state_dict(sd)
        print('Direct load OK')
        return
    except Exception:
        print('Direct load failed, using key mapping...')

    # Weight mapping: body1/2/3 → body.0/.1/.2/...
    md = model.state_dict()
    new = collections.OrderedDict()
    idx = 0
    for group, count in [('body1', 3), ('body2', 4), ('body3', 14)]:
        for i in range(count):
            new[f'{group}.{i}.'] = f'body.{idx}.'
            idx += 1

    for k, v in sd.items():
        k_clean = k.replace('module.', '')
        mapped_k = k_clean
        for old_prefix, new_prefix in new.items():
            if k_clean.startswith(old_prefix):
                mapped_k = k_clean.replace(old_prefix, new_prefix)
                break
        if mapped_k in md and md[mapped_k].size() == v.size():
            md[mapped_k] = v

    model.load_state_dict(md)
    print('Weight mapping complete')


EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']
NUM_CLASSES = len(EMOTIONS)

model = POSTER(num_classes=NUM_CLASSES, depth=8)

checkpoint_path = 'outputs/models/poster_best.pth'
if os.path.exists(checkpoint_path):
    load_poster_weights(model, checkpoint_path, device)
    print(f'Loaded: {checkpoint_path}')
else:
    print(f'File not found: {checkpoint_path}')

model.to(device)
model.eval()

In [ ]:
# ─── Grad-CAM implementation ───
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.features = None
        self.gradients = None

        # Forward hook: capture feature maps
        target_layer.register_forward_hook(self._forward_hook)
        # Backward hook: capture gradients
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        self.features = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def compute(self, x, class_idx=None):
        """
        Compute Grad-CAM heatmap.
        Returns: heatmap (H, W), pred_class, confidence
        """
        # Forward
        logits, _ = self.model(x)
        probs = F.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = torch.argmax(logits[0]).item()

        pred_class = class_idx
        confidence = probs[0, pred_class].item()

        # Zero gradients
        self.model.zero_grad()

        # Backward: class score
        score = logits[:, class_idx]
        score.backward()

        # Global Average Pooling trên gradients → weights
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]

        # Weighted combination
        cam = (weights * self.features).sum(dim=1)  # [1, H, W]
        cam = F.relu(cam)  # chỉ giữ positive influence

        # Normalize
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)

        return cam[0].detach().cpu().numpy(), pred_class, confidence, probs[0].detach().cpu().numpy()


# Target: last bottleneck_IR module trong IR-50 body
target_layer = model.ir_back.body[-1]
print(f'Target layer: {target_layer}')

gradcam = GradCAM(model, target_layer)

In [ ]:
# ─── Image transform ───
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def load_image(path):
    img = Image.open(path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    img_tensor.requires_grad_()
    return img, img_tensor


# ─── Chọn 2 ảnh mẫu ───
TEST_DIR = 'data/DATASET/test'
sample_images = [
    os.path.join(TEST_DIR, '4', 'test_0003_aligned.jpg'),  # Happiness
    os.path.join(TEST_DIR, '1', 'test_0002_aligned.jpg'),  # Surprise
]

for p in sample_images:
    assert os.path.exists(p), f'Missing: {p}'
print('All sample images found.')

In [ ]:
# ─── Visualization ───
COLORMAP = 'jet'


def visualize_gradcam(img_pil, heatmap, pred_idx, conf, true_label=None):
    img_np = np.array(img_pil)
    h, w = img_np.shape[:2]

    # Resize heatmap về kích thước ảnh gốc
    heatmap_img = np.array(Image.fromarray(heatmap).resize((w, h), Image.BILINEAR))

    fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))

    title = f"Pred: {EMOTIONS[pred_idx]} ({conf:.2%})"
    if true_label is not None:
        title += f' | True: {true_label}'
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # 1. Original
    axes[0].imshow(img_np)
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')

    # 2. Heatmap raw
    im = axes[1].imshow(heatmap_img, cmap=COLORMAP, interpolation='bilinear')
    axes[1].set_title('Grad-CAM Heatmap', fontsize=12)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)

    # 3. Overlay (alpha thấp)
    axes[2].imshow(img_np, alpha=0.5)
    im = axes[2].imshow(heatmap_img, cmap=COLORMAP, alpha=0.5, interpolation='bilinear')
    axes[2].set_title('Overlay (α=0.5)', fontsize=12)
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)

    # 4. Overlay (alpha cao)
    axes[3].imshow(img_np, alpha=0.3)
    im = axes[3].imshow(heatmap_img, cmap=COLORMAP, alpha=0.7, interpolation='bilinear')
    axes[3].set_title('Overlay (α=0.7)', fontsize=12)
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046)

    plt.tight_layout()
    plt.show()


def visualize_gradcam_with_contour(img_pil, heatmap, pred_idx, conf, true_label=None):
    """Phiên bản có contour lines — highlight rõ vùng quan trọng."""
    img_np = np.array(img_pil)
    h, w = img_np.shape[:2]
    heatmap_img = np.array(Image.fromarray(heatmap).resize((w, h), Image.BILINEAR))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    title = f"Pred: {EMOTIONS[pred_idx]} ({conf:.2%})"
    if true_label is not None:
        title += f' | True: {true_label}'
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Overlay + contour
    axes[0].imshow(img_np, alpha=0.5)
    axes[0].imshow(heatmap_img, cmap=COLORMAP, alpha=0.5, interpolation='bilinear')
    contour = axes[0].contour(heatmap_img, levels=5, colors='white', linewidths=0.8, alpha=0.7)
    axes[0].clabel(contour, inline=True, fontsize=8, fmt='%.2f')
    axes[0].set_title('Grad-CAM + Contour', fontsize=12)
    axes[0].axis('off')

    # Binary mask: threshold 50%
    mask = heatmap_img > 0.5
    masked = img_np.copy()
    highlight = img_np.copy()
    highlight[mask] = [255, 0, 0]  # đỏ
    blended = cv2.addWeighted(img_np, 0.6, highlight, 0.4, 0)
    axes[1].imshow(blended)
    axes[1].set_title('Vùng >50% activation (đỏ)', fontsize=12)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Run trên 2 ảnh ───
import cv2

for img_path in sample_images:
    true_class = os.path.basename(os.path.dirname(img_path))
    true_label = EMOTIONS[int(true_class) - 1]

    img_pil, img_tensor = load_image(img_path)
    heatmap, pred_idx, conf, probs = gradcam.compute(img_tensor)

    print(f"{'='*60}")
    print(f"Image: {os.path.basename(img_path)}")
    print(f"True: {true_label} | Predicted: {EMOTIONS[pred_idx]} ({conf:.2%})")
    print(f"Probabilities:")
    for i, p in enumerate(probs):
        bar = '█' * int(p * 30)
        print(f"  {EMOTIONS[i]:12s}: {p:.2%} {bar}")

    visualize_gradcam(img_pil, heatmap, pred_idx, conf, true_label)
    visualize_gradcam_with_contour(img_pil, heatmap, pred_idx, conf, true_label)

## Nhận xét

- Vùng đỏ/cam trên heatmap = nơi IR-50 backbone tập trung khi dự đoán
- Model tốt sẽ focus vào mắt, miệng, lông mày — đặc trưng cảm xúc chính
- Nếu model focus ra background → có vấn đề về học